In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

from src.models import SIRM, SIRT, SIRV
from helps import *




from src.utils.batch_sweep import sweep_two_parameters

# Import the matrix creation function
from src.utils.Contact_Matrix import create_contact_matrix

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.gridspec import GridSpec

def simple_parameter_grid(data_matrices, nrows, ncols, figsize=(10, 8), 
                         cmap='viridis', vmin=0, vmax=1, extent=[0, 1, 0, 6]):
    """
    Super duper simple version - just plot matrices with colorbars
    
    Args:
        data_matrices: List of 2D arrays to plot
        nrows, ncols: Grid dimensions
        figsize: Figure size
        cmap: Colormap
        vmin, vmax: Color scale limits
        
    Returns:
        fig, axes
    """
    # Create figure with extra column for colorbars
    fig = plt.figure(figsize=figsize)
    gs = GridSpec(nrows, ncols + 1, figure=fig, width_ratios=[1] * ncols + [0.05])
    
    # Create axes
    axes = np.empty((nrows, ncols), dtype=object)
    for i in range(nrows):
        for j in range(ncols):
            axes[i, j] = fig.add_subplot(gs[i, j])
    
    cbar_axes = []
    for i in range(nrows):
        cbar_axes.append(fig.add_subplot(gs[i, -1]))
    
    # Plot matrices
    for i in range(nrows):
        for j in range(ncols):
            idx = i * ncols + j
            
            if idx < len(data_matrices):
                data = data_matrices[idx]
                im = axes[i, j].imshow(
                    np.flipud(data), 
                    cmap=cmap, 
                    vmin=vmin, 
                    vmax=vmax,
                    aspect='auto',
                    extent=extent
                )
                
                # Add colorbar for first column of each row
                if j == 0:
                    fig.colorbar(im, cax=cbar_axes[i])
    
    return fig, axes, cbar_axes


my_map = discretize_cmaps("hot_r",21)
my_map.set_bad(color='gray')

In [ ]:
temp = read_json("./parameters.json")
mus, taus, xis, PARAMS = temp["mus"], temp["taus"], temp["xis"], temp["PARAMS"]



# Bootstrap figure

In [ ]:
# calculate and print the range of the confidence interval:

from bootstrap_CM_module import *
df = pd.read_csv("data_homophily.csv")
df = df.dropna()


R_M = bootstrap_mph(df, "masks", n_bootstrap=20000)
R_T = bootstrap_mph(df, "testing", n_bootstrap=20000)
R_V = bootstrap_mph(df, "vacc", n_bootstrap=20000)

print(f"[{np.round(R_M['M_CI'], 2)}, {np.round(R_M['P_CI'], 2)}, {np.round(R_M['H_CI'], 2)}]")
print(f"[{np.round(R_T['M_CI'], 2)}, {np.round(R_T['P_CI'], 2)}, {np.round(R_T['H_CI'], 2)}]")
print(f"[{np.round(R_V['M_CI'], 2)}, {np.round(R_V['P_CI'], 2)}, {np.round(R_V['H_CI'], 2)}]")

In [ ]:
fig, axes = plot_bootstrap_heatmap(R_M, R_T, R_V, bins = 150, figsize=(7.09, 7))
axes[(0,0)].set_title("Masks", fontsize=12)
axes[(0,1)].set_title("Testing", fontsize=12)
axes[(0,2)].set_title("Vaccines", fontsize=12)
for i in range(3):
    axes[(0,i)].set_xlabel("Mean", fontsize=10)
for i in range(2):
    axes[(i,0)].set_ylabel("Homophily", fontsize=10)
axes[(2,0)].set_ylabel("Mean", fontsize=10)
for i in range(3):
    axes[(2,i)].set_xlabel("Polarization", fontsize=10)


for i in range(3):
    axes[(0,i)].set_xlim(0.4, 0.85)
    axes[(1,i)].set_xlim(0.2, 0.65)
    axes[(2,i)].set_xlim(0.2, 0.65)

    axes[(0,i)].set_xticks([0.4, 0.6, 0.8])
    axes[(1,i)].set_xticks([0.2, 0.4,  0.6])
    axes[(2,i)].set_xticks([0.2, 0.4,  0.6])

for i in range(3):
    axes[(0,i)].set_ylim(1.25,3.75)
    axes[(1,i)].set_ylim(1.25,3.75)
    axes[(2,i)].set_ylim(0.4,0.85)
    axes[(2,i)].set_yticks([0.4, 0.6, 0.8])


fig.savefig("Figures/Fig_SI/Fig_SI_bootstrapping.pdf", bbox_inches='tight')

# Number of Compartments figure

In [ ]:
N_compartments = [3, 5, 10, 30, 100]
RM = []
RT = []
RV = []
NB = 100
NP = 100

homophilic_tendency = {"m": 0, "M": 6, "n": NB}
pol_range = {"m": 0, "M": 1, "n": NP}

for N in N_compartments:
    res = sweep_two_parameters(
        model_module=SIRM,
        param1_name="beta_params",           # parameter 1 name
        param1_range=pol_range,    # parameter 1 range
        param2_name="homophilic_tendency",      # parameter 2 name
        param2_range=homophilic_tendency,         # parameter 2 range
        custom_base_params=PARAMS,
        simulated_days=1000,
        population_size=N,
        batch_size=1000
    )
    RM.append(res)
    res = sweep_two_parameters(
        model_module=SIRT,
        param1_name="beta_params",           # parameter 1 name
        param1_range=pol_range,    # parameter 1 range
        param2_name="homophilic_tendency",      # parameter 2 name
        param2_range=homophilic_tendency,         # parameter 2 range
        custom_base_params=PARAMS,
        simulated_days=1000,
        population_size=N,
        batch_size=1000
    )
    RT.append(res)
    res = sweep_two_parameters(
        model_module=SIRV,
        param1_name="beta_params",           # parameter 1 name
        param1_range=pol_range,    # parameter 1 range
        param2_name="homophilic_tendency",      # parameter 2 name
        param2_range=homophilic_tendency,         # parameter 2 range
        custom_base_params=PARAMS,
        simulated_days=1000,
        population_size=N,
        batch_size=1000
    )
    RV.append(res)

    print(f"Finished {N} compartments")

In [ ]:
# concatenate all results
results = RM + RT + RV

matrices = []
for i in range(len(results)):
    TI = np.sum(results[i]["final_state"]["R"], axis=2) + np.sum(results[i]["final_state"]["I"], axis=2)
    matrices.append(TI)

fig, plot_axes, cbar_axes = simple_parameter_grid(matrices, 3, 5, cmap=my_map, vmin=0, vmax=1, figsize=(7.09, 6.09))
plt.subplots_adjust(hspace=0.3, wspace=0.2)


for i in range(3):
    for j in range(5):
        plot_axes[i, j].set_xticks([0, 0.5, 1])
        plot_axes[i, j].set_yticks([0, 3, 6])
        plot_axes[i, j].set_xticklabels([0, 0.5, 1])
        

        if j > 0:
            plot_axes[i, j].set_yticklabels([])
        else:
            plot_axes[i, j].set_yticklabels([0, 3, 6])

        if i < 2:
            plot_axes[i, j].set_xticklabels([])
        else:
            plot_axes[i, j].set_xticklabels([0, 0.5, 1])


plot_axes[0, 0].set_ylabel("SIRM\n\nHomophily", fontsize=14)
plot_axes[1, 0].set_ylabel("SIRT\n\nHomophily", fontsize=14)
plot_axes[2, 0].set_ylabel("SIRV\n\nHomophily", fontsize=14)

plot_axes[2, 2].set_xlabel("Polarization", fontsize=14)


cbar_axes[0].set_yticks([0, 0.5, 1])
cbar_axes[1].set_ylabel("Fraction of Infected", fontsize=14)
cbar_axes[1].set_yticks([0, 0.5, 1])
cbar_axes[2].set_yticks([0, 0.5, 1])


plot_axes[0, 0].set_title("N = " + str(N_compartments[0]), fontsize=14)
plot_axes[0, 1].set_title("N = " + str(N_compartments[1]), fontsize=14)
plot_axes[0, 2].set_title("N = " + str(N_compartments[2]), fontsize=14)
plot_axes[0, 3].set_title("N = " + str(N_compartments[3]), fontsize=14)
plot_axes[0, 4].set_title("N = " + str(N_compartments[4]), fontsize=14)


fig.savefig("figures/Fig_SI/Dependency_on_NCOMPARTMENTS.pdf", dpi=150, bbox_inches='tight')

# min-maxing

## SIR-M

In [ ]:
MU_m = [0. , 0.2, 0.4]
MU_M = [0.6, 0.8, 1.]


RES_M_mumu = []


for i in range(3):
    for j in range(3):
        CB =PARAMS.copy()
        CB["mu_min"] = MU_m[i]
        CB["mu_max"] = MU_M[j]
        res = sweep_two_parameters(
            model_module=SIRM,
            param1_name="beta_params",           # parameter 1 name
            param1_range=pol_range,    # parameter 1 range
            param2_name="homophilic_tendency",      # parameter 2 name
            param2_range=homophilic_tendency,         # parameter 2 range
            custom_base_params=CB,
            simulated_days=1000,
            population_size=20,
            batch_size=1000
        )
        RES_M_mumu.append(res)


In [ ]:
matrices = []
for i in range(len(RES_M_mumu)):
    TI = np.sum(RES_M_mumu[i]["final_state"]["R"], axis=2) + np.sum(RES_M_mumu[i]["final_state"]["I"], axis=2)
    matrices.append(TI)

fig, plot_axes, cbar_axes = simple_parameter_grid(matrices, 3, 3, cmap=my_map, vmin=0, vmax=1, figsize=(7.09, 6.09))
plt.subplots_adjust(hspace=0.3, wspace=0.2)


for i in range(3):
    for j in range(3):
        plot_axes[i, j].set_xticks([0, 0.5, 1])
        plot_axes[i, j].set_yticks([0, 3, 6])
        plot_axes[i, j].set_xticklabels([0, 0.5, 1])
        plot_axes[i, j].set_yticklabels([0, 3, 6])
        if i < 2:
            plot_axes[i, j].set_xticklabels([])
        if j > 0:
            plot_axes[i, j].set_yticklabels([])
    cbar_axes[i].set_yticks([0, 0.5, 1])
    plot_axes[i, 0].set_ylabel(r"$\mu_{min} = " + f"{MU_m[i]}" + r"$" + "\n\nHomophily", fontsize=14)
    plot_axes[2, i].set_xlabel("Polarization", fontsize=14)
    plot_axes[0, i].set_title(r"$\mu_{max} = " + f"{MU_M[i]}" + r"$", fontsize=14)

cbar_axes[1].set_ylabel("Fraction of Infected", fontsize=14)

fig.savefig("figures/Fig_SI/MIN-MAX_SIRM.pdf", dpi=150, bbox_inches='tight')

## SIR-T

In [ ]:
TAU_m = [0, 0.025, 0.05]
TAU_M = [0.1 , 0.2,  0.3]

CB =PARAMS.copy()
RES_T_tautau = []
for i in range(3):
    for j in range(3):
        CB["testing_rate_min"] = TAU_m[i]
        CB["testing_rate_max"] = TAU_M[j]
        results_TESTS_00 = sweep_two_parameters(
            model_module=SIRT,
            param1_name="beta_params",           # parameter 1 name
            param1_range=pol_range,    # parameter 1 range
            param2_name="homophilic_tendency",      # parameter 2 name
            param2_range=homophilic_tendency,         # parameter 2 range
            custom_base_params=CB,
            simulated_days=1000,
            population_size=20,
            batch_size=1000
        )
        RES_T_tautau.append(results_TESTS_00)

In [ ]:
matrices = []
for i in range(len(RES_T_tautau)):
    TI = np.sum(RES_T_tautau[i]["final_state"]["R"], axis=2) + np.sum(RES_T_tautau[i]["final_state"]["I"], axis=2)
    matrices.append(TI)

fig, plot_axes, cbar_axes = simple_parameter_grid(matrices, 3, 3, cmap=my_map, vmin=0, vmax=1, figsize=(7.09, 6.09))
plt.subplots_adjust(hspace=0.3, wspace=0.2)

count = 0
for i in range(3):
    for j in range(3):
        plot_axes[i, j].set_xticks([0, 0.5, 1])
        plot_axes[i, j].set_yticks([0, 3, 6])
        plot_axes[i, j].set_xticklabels([0, 0.5, 1])
        plot_axes[i, j].set_yticklabels([0, 3, 6])
        if i < 2:
            plot_axes[i, j].set_xticklabels([])
        if j > 0:
            plot_axes[i, j].set_yticklabels([])
        count += 1

    cbar_axes[i].set_yticks([0, 0.5, 1])
    plot_axes[i, 0].set_ylabel(r"$\tau_{min} = " + f"{TAU_m[i]}" + r"$" + "\n\nHomophily", fontsize=14)
    plot_axes[2, i].set_xlabel("Polarization", fontsize=14)
    plot_axes[0, i].set_title(r"$\tau_{max} = " + f"{TAU_M[i]}" + r"$", fontsize=14)
    if i == 1:
        cbar_axes[i].set_ylabel("Fraction ofInfected", fontsize=14)
    



fig.savefig("figures/Fig_SI/MIN-MAX_SIRT.pdf", dpi=150, bbox_inches='tight')

## SIR-V

In [ ]:
XIS_m = np.array([0.   , 0.002, 0.004])
XIS_M = np.array([0.006, 0.008 , 0.01])

CB =PARAMS.copy()
RES_XIS_xisxis = []
for i in range(3):
    for j in range(3):
        CB["vaccination_rate_min"] = XIS_m[i]
        CB["vaccination_rate_max"] = XIS_M[j]
        results_VACCINATION_00 = sweep_two_parameters(
            model_module=SIRV,
            param1_name="beta_params",           # parameter 1 name
            param1_range=pol_range,    # parameter 1 range
            param2_name="homophilic_tendency",      # parameter 2 name
            param2_range=homophilic_tendency,         # parameter 2 range
            custom_base_params=CB,
            simulated_days=1000,
            population_size=20,
            batch_size=1000
        )
        RES_XIS_xisxis.append(results_VACCINATION_00)

In [ ]:
matrices = []
for i in range(len(RES_XIS_xisxis)):
    TI = np.sum(RES_XIS_xisxis[i]["final_state"]["R"], axis=2) + np.sum(RES_XIS_xisxis[i]["final_state"]["I"], axis=2)
    matrices.append(TI)
fig, plot_axes, cbar_axes = simple_parameter_grid(matrices, 3, 3, cmap=my_map, vmin=0, vmax=1, figsize=(7.09, 6.09))
plt.subplots_adjust(hspace=0.3, wspace=0.2)

count = 0
for i in range(3):
    for j in range(3):
        plot_axes[i, j].set_xticks([0, 0.5, 1])
        plot_axes[i, j].set_yticks([0, 3, 6])
        plot_axes[i, j].set_xticklabels([0, 0.5, 1])
        plot_axes[i, j].set_yticklabels([0, 3, 6])
        if i < 2:
            plot_axes[i, j].set_xticklabels([])
        if j > 0:
            plot_axes[i, j].set_yticklabels([])
        #plot_axes[i, j].set_title(f"tau_min = {TAU_m[j]}", fontsize=14)
        count += 1

    cbar_axes[i].set_yticks([0, 0.5, 1])
    plot_axes[i, 0].set_ylabel(r"$\xi_{min} = " + f"{XIS_m[i]}" + r"$" + "\n\nHomophily", fontsize=14)
    plot_axes[2, i].set_xlabel("Polarization", fontsize=14)
    plot_axes[0, i].set_title(r"$\xi_{max} = " + f"{XIS_M[i]}" + r"$", fontsize=14)
    if i == 1:
        cbar_axes[i].set_ylabel("Fraction ofInfected", fontsize=14)


fig.savefig("figures/Fig_SI/MIN-MAX_SIRV.pdf.pdf", dpi=150, bbox_inches='tight')




# how timely is the vaccination campaign?

In [ ]:
I0s = np.array([1e-6, 1e-5, 1e-4, 1e-3])
XIS_M = np.array([0.006, 0.008 , 0.01])

CB =PARAMS.copy()
RES_XIS_xisxis_I0s = []
for i in range(4):
    for j in range(3):
        CB["vaccination_rate_min"] = 0
        CB["vaccination_rate_max"] = XIS_M[j]
        
        results_VACCINATION_00 = sweep_two_parameters(
            model_module=SIRV,
            param1_name="beta_params",           # parameter 1 name
            param1_range=pol_range,    # parameter 1 range
            param2_name="homophilic_tendency",      # parameter 2 name
            param2_range=homophilic_tendency,         # parameter 2 range
            custom_base_params=CB,
            simulated_days=1000,
            population_size=20,
            batch_size=1000,
            initial_infected_prop=I0s[i]
        )
        RES_XIS_xisxis_I0s.append(results_VACCINATION_00)

In [ ]:
matrices = []
for i in range(len(RES_XIS_xisxis_I0s)):
    TI = np.sum(RES_XIS_xisxis_I0s[i]["final_state"]["R"], axis=2) + np.sum(RES_XIS_xisxis_I0s[i]["final_state"]["I"], axis=2)
    matrices.append(TI)
fig, plot_axes, cbar_axes = simple_parameter_grid(matrices, 4, 3, cmap=my_map, vmin=0, vmax=1, figsize=(7.09, 6.09))
plt.subplots_adjust(hspace=0.3, wspace=0.2)

count = 0
for i in range(4):
    for j in range(3):
        plot_axes[i, j].set_xticks([0, 0.5, 1])
        plot_axes[i, j].set_yticks([0, 3, 6])
        if i < 3:
            plot_axes[i, j].set_xticklabels([])
        else:
            plot_axes[i, j].set_xticklabels([0, 0.5, 1])
        if j > 0:
            plot_axes[i, j].set_yticklabels([])
        else:
            plot_axes[i, j].set_yticklabels([0, 3, 6])
        count += 1

    cbar_axes[i].set_yticks([0, 0.5, 1])
    plot_axes[i, 0].set_ylabel(r"$I_0 = 10^{" + f"{int(np.log10(I0s[i]))}" + r"}$" + "\n\nHomophily", fontsize=14)
    if i == 1:
        cbar_axes[i].set_ylabel("Fraction ofInfected", fontsize=14)

for j in range(3):
    plot_axes[3, j].set_xlabel("Polarization", fontsize=14)
    plot_axes[0, j].set_title(r"$\xi_{max} = " + f"{XIS_M[j]}" + r"$", fontsize=14)


fig.savefig("figures/Fig_SI/SIRV_I0.pdf", dpi=150, bbox_inches='tight')

# Pol and mean

In [ ]:
mean_range = {"m": 0, "M": 1, "n": 100}
H = [0, 3, 6]
RES = []


for i in range(3):
    CB =PARAMS.copy()
    CB["homophilic_tendency"] = H[i]

    temp = sweep_two_parameters(
        model_module=SIRM,
        param1_name="beta_params",           # polarization parameter
        param1_range=pol_range,    
        param2_name="mean",                  # average behavior parameter
        param2_range=mean_range,             # range for mean behavior (0 to 1)
        custom_base_params=CB,
        simulated_days=1000,
        population_size=20,
        batch_size=1000
    )
    RES.append(temp)
    temp = sweep_two_parameters(
        model_module=SIRT,
        param1_name="beta_params",           # polarization parameter
        param1_range=pol_range,    
        param2_name="mean",                  # average behavior parameter
        param2_range=mean_range,             # range for mean behavior (0 to 1)
        custom_base_params=CB,
        simulated_days=1000,
        population_size=20,
        batch_size=1000
    )
    RES.append(temp)
    temp = sweep_two_parameters(
        model_module=SIRV,
        param1_name="beta_params",           # polarization parameter
        param1_range=pol_range,    
        param2_name="mean",                  # average behavior parameter
        param2_range=mean_range,             # range for mean behavior (0 to 1)
        custom_base_params=CB,
        simulated_days=1000,
        population_size=20,
        batch_size=1000
    )
    RES.append(temp)

In [ ]:
matrices = []
for i in range(len(RES)):
    TI = np.sum(RES[i]["final_state"]["R"], axis=2) + np.sum(RES[i]["final_state"]["I"], axis=2)
    matrices.append(TI)

fig, plot_axes, cbar_axes = simple_parameter_grid(matrices, 3, 3, cmap=my_map, vmin=0, vmax=1, figsize=(7.09, 6.09), extent=[0, 1, 0, 1])
plt.subplots_adjust(hspace=0.3, wspace=0.2)

homophily = [0, 3, 6]

for i in range(3):
    for j in range(3):
        plot_axes[i, j].set_xticks([0, 0.5, 1])
        plot_axes[i, j].set_yticks([0, 0.5, 1])
        plot_axes[i, j].set_xticklabels([0, 0.5, 1])
        plot_axes[i, j].set_yticklabels([0, 0.5, 1])
        if i < 2:
            plot_axes[i, j].set_xticklabels([])
        if j > 0:
            plot_axes[i, j].set_yticklabels([])
    plot_axes[0, i].set_title(f"Homophily = {homophily[i]}", fontsize=14)


    cbar_axes[i].set_yticks([0, 0.5, 1])
    plot_axes[2, i].set_xlabel("Polarization", fontsize=14)
    if i == 1:
        cbar_axes[i].set_ylabel("Fraction ofInfected", fontsize=14)

plot_axes[0, 0].set_ylabel("SIRM\n\nMean", fontsize=14)
plot_axes[1, 0].set_ylabel("SIRT\n\nMean", fontsize=14)
plot_axes[2, 0].set_ylabel("SIRV\n\nMean", fontsize=14)



fig.savefig("figures/Fig_SI/mean_comparison.pdf", dpi=150, bbox_inches='tight')

In [ ]:
def find_critical_homophily(res, threshold=0):
    """
    For each beta_M result, find the max homophily where Jensen effect holds.
    
    res: result from sweep_two_parameters
    Returns: array of critical h values, shape (n_betas,)
    
    The grid is (n_pol x n_hom), R has shape (n_hom, n_pol, pop_size).
    Jensen works at a given h if dR/d_pol < threshold (increasing pol -> fewer cases).
    """
    R = np.array(res["final_state"]["R"])          # (n_hom, n_pol, pop_size)
    h_vals = np.array(res["parameter_grid"]["param2_vals"][:, 0])  # (n_hom,)
    
    total_R = R.sum(axis=-1)   # (n_hom, n_pol) - total recovered = total cases
    
    # Slope of R along polarization axis for each h
    slopes = np.diff(total_R, axis=1)  # (n_hom, n_pol-1)
    
    # Jensen works at h if ALL slopes are negative (monotonically decreasing)
    jensen_holds = np.all(slopes < threshold, axis=1)  # (n_hom,)
    
    # Find max h where Jensen holds
    valid_h = h_vals[jensen_holds]
    return valid_h.max() if len(valid_h) > 0 else 0.0

NB = 100
NP = 100
homophilic_tendency = {"m": 0, "M": 6, "n": NB}
pol_range = {"m": 0, "M": 1, "n": NP}

print("### SIR-M ###")
PS = 50
res_list_M_slim = []
betas = np.linspace(0.15, 0.3, 30)
for b in betas:
    PARAMS["beta_M"] = b
    M = sweep_two_parameters(
        model_module=SIRM,
        param1_name="beta_params",           # parameter 1 name
        param1_range=pol_range,    # parameter 1 range
        param2_name="homophilic_tendency",      # parameter 2 name
        param2_range=homophilic_tendency,         # parameter 2 range
        custom_base_params=PARAMS,
        simulated_days=500,
        population_size=PS,
        batch_size=2000
    )
    res_list_M_slim.append(M)
    print(f"Completed beta_M = {b}")



In [ ]:
Lxinches = 178 / 25.4
Lyinches = 178 / 25.4

critical_h_M = [find_critical_homophily(res) for res in res_list_M_slim]

fig, ax = plt.subplots(figsize=(Lxinches, Lyinches/2.5))
ax.plot(betas*10, critical_h_M, 'o-')
ax.set_xlabel('$R_0^*$')
ax.set_ylabel('Critical homophily')
ax.set_title('Critical homophily vs $R_0^*$ for SIR-M')

ax.set_xticks([1.5, 2.0, 2.5, 3.0])
ax.set_yticks([0, 3, 6])
ax.set_yticklabels(['0', '3', '6+'])

ax.set_xlim(1.48, 3.02)
ax.set_ylim(-0.11, 6.11)

# remove top and right spines

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

fig.tight_layout()
fig.savefig("figures/Fig_SI/critical_homophily_SIR-M.pdf", dpi=300, bbox_inches='tight')

# General comparison for Fig 4

In [ ]:
from copy import deepcopy

def run_one(R0, PARAMS, spec, model, pol_baseline, hom_baseline, simulated_days = 10000, sampling_points = 2):
    CB = deepcopy(PARAMS)
    
    pols = {"m": spec["pol"][0], "M": spec["pol"][2], "n": sampling_points}
    homs = {"m": spec["h"][0], "M": spec["h"][2], "n": sampling_points}


    CB["beta_M"] = R0 * CB["recovery_rate"]
    CB["fixed_mean"] = spec["mean"][1]
    # run the heterogeneous simulation
    RES_HET = sweep_two_parameters(
        model_module=model,
        param1_name="beta_params",              # parameter 1 name
        param1_range=pols,                      # parameter 1 range
        param2_name="homophilic_tendency",      # parameter 2 name
        param2_range=homs,                      # parameter 2 range
        custom_base_params=CB,
        simulated_days=simulated_days,
        population_size=5,
        batch_size= sampling_points*sampling_points
    )
    # run the homogeneous simulation
    RES_HOM = sweep_two_parameters(
        model_module=model,
        param1_name="beta_params",              # parameter 1 name
        param1_range=pol_baseline,              # parameter 1 range
        param2_name="homophilic_tendency",      # parameter 2 name
        param2_range=hom_baseline,              # parameter 2 range
        custom_base_params=CB,
        simulated_days=simulated_days,
        population_size=5,
        batch_size=1
    )
    HET = np.sum(RES_HET["final_state"]["R"], axis=2) + np.sum(RES_HET["final_state"]["I"], axis=2)
    HOM = np.sum(RES_HOM["final_state"]["R"], axis=2) + np.sum(RES_HOM["final_state"]["I"], axis=2)
    return HET, HOM

In [ ]:
R0s = np.linspace(1, 5, 100)
sirm_min = np.zeros((len(R0s), 3, 3, 3))
sirm_max = np.zeros((len(R0s),3,3, 3))
sirm_hom = np.zeros((len(R0s),3,3, 3))

sirt_min = np.zeros((len(R0s),3,3, 3))
sirt_max = np.zeros((len(R0s),3,3, 3))
sirt_hom = np.zeros((len(R0s),3,3, 3))

sirv_min = np.zeros((len(R0s),3,3, 3))
sirv_max = np.zeros((len(R0s),3,3, 3))
sirv_hom = np.zeros((len(R0s),3,3, 3))



pol_l = [0.2, 0.45, 0.7]
pol_h = [0.3, 0.55, 0.8]

hom_l = [0.5 , 2.5, 4.5]
hom_h = [1, 3, 5]


P = [[0.2, 0.25, 0.3],
      [1/3-0.05, 1/3, 1/3+0.05],
      [0.45, 0.5, 0.55]]


H = [[0.75, 1, 1.25],
      [2.75, 3, 3.25],
      [4.75, 5, 5.25]]

M = [[0.3, 0.3, 0.3],[0.5, 0.5,0.5],[0.7, 0.7,0.7]]


for k in range(3):
    for j in range(3):
        for m in range(3):
            pars = {"mean": M[m], "pol" : P[k], "h" : H[j]}
            for i, R0 in enumerate(R0s):
                  het, hom = run_one(R0, PARAMS, pars, SIRM, [0.001], [0], simulated_days=1000, sampling_points=2)
                  sirm_min[i][k][j][m] = np.min(np.array(het))
                  sirm_max[i][k][j][m] = np.max(np.array(het))
                  sirm_hom[i][k][j][m] = hom[0][0]

                  het, hom = run_one(R0, PARAMS, pars, SIRT, [0.001], [0], simulated_days=1000, sampling_points=2)
                  sirt_min[i][k][j][m] = np.min(np.array(het))
                  sirt_max[i][k][j][m] = np.max(np.array(het))
                  sirt_hom[i][k][j][m] = hom[0][0]

                  het, hom = run_one(R0, PARAMS, pars, SIRV, [0.001], [0], simulated_days=1000, sampling_points=2)
                  sirv_min[i][k][j][m] = np.min(np.array(het))
                  sirv_max[i][k][j][m] = np.max(np.array(het))
                  sirv_hom[i][k][j][m] = hom[0][0]

                  print(f"Completed R0 = {R0}, pol = {P[k]}, hom = {H[j]}, mean = {M[m]}")




In [ ]:
from src.utils.distributions import pol_mean_to_ab, my_beta_asymmetric

colors_m = ['#bdc9e1','#74a9cf','#0570b0']
colors_t = ['#d7b5d8','#df65b0','#ce1256']
colors_v = ['#b2e2e2','#66c2a4','#238b45']



Lxinches = 178 / 25.4
Lyinches = 178 / 25.4

fig = plt.figure(figsize=(Lxinches, Lyinches))
gs = fig.add_gridspec(2, 1, height_ratios=[0.15, 1], hspace=0.1)

gs_top = gs[0].subgridspec(1, 3, wspace=0.1)
gs_bot = gs[1].subgridspec(3, 3, hspace=0.3, wspace=0.3)

grays = ['#d9d9d9', '#969696', '#525252']

for k in range(3):
    gs_hist = gs_top[k].subgridspec(1, 3, wspace=0.05)
    for MM in range(3):
        ax_hist = fig.add_subplot(gs_hist[MM])
        a, b = pol_mean_to_ab(np.array(P[k][1]), np.array(M[MM][1]))
        n_bins = 5
        x = np.linspace(0, 1, n_bins)
        y = my_beta_asymmetric(a, b, n_bins)
        ax_hist.bar(x, y, width=1/n_bins*0.8, color=grays[MM], edgecolor='black', linewidth=0.3)
        ax_hist.set_xlim(-0.1, 1.1)
        ax_hist.set_xticks([])
        ax_hist.set_yticks([])
        if MM == 1:
            ax_hist.set_title(f"pol={P[k][1]:.2f}", fontsize=8)
        ax_hist.set_xlabel(f"m={M[MM][1]:.1f}", fontsize=8)

axs = [[fig.add_subplot(gs_bot[j, k]) for k in range(3)] for j in range(3)]

for MM in range(3):
    for k in range(3):
        for j in range(3):
            M_avg = ((sirm_min[:,k,j,MM] + sirm_max[:,k,j,MM]) / 2 - sirm_hom[:,k,j,MM]) * 100
            T_avg = ((sirt_min[:,k,j,MM] + sirt_max[:,k,j,MM]) / 2 - sirt_hom[:,k,j,MM]) * 100
            V_avg = ((sirv_min[:,k,j,MM] + sirv_max[:,k,j,MM]) / 2 - sirv_hom[:,k,j,MM]) * 100

            LW = 1.5
            if MM == 2:
                axs[j][k].plot(R0s, M_avg, color=colors_m[MM], linewidth=LW, linestyle='-', label = "SIR-M")
                axs[j][k].plot(R0s, T_avg, color=colors_t[MM], linewidth=LW, linestyle='--', label = "SIR-T")
                axs[j][k].plot(R0s, V_avg, color=colors_v[MM], linewidth=LW, linestyle=':', label = "SIR-V")
            else:
                axs[j][k].plot(R0s, M_avg, color=colors_m[MM], linewidth=LW, linestyle='-')
                axs[j][k].plot(R0s, T_avg, color=colors_t[MM], linewidth=LW, linestyle='--')
                axs[j][k].plot(R0s, V_avg, color=colors_v[MM], linewidth=LW, linestyle=':')
            axs[j][k].axhline(0, color='black', linestyle='--', linewidth=1)
            axs[j][k].axvline(2.5, color='black', linestyle=':', linewidth=1)
            axs[j][k].set_ylim(-50, 50)
            axs[j][k].set_xlim(1, 5)
            axs[j][k].set_xticks([1, 3, 5])
            axs[j][k].set_yticks([-50, 0, 50])

            if k == 0:
                axs[j][k].set_ylabel(f"hom={H[j][1]:.0f}", fontsize=10)
                axs[j][k].text(-0.25, 0.5, "Structural bias  $\\Delta I$", fontsize=8, transform=axs[j][k].transAxes, ha='center', va='center', rotation=90)
            if j == 2:
                axs[j][k].set_xlabel("$R_0^*$", fontsize=10)
            axs[j][k].legend(fontsize=6)

fig.canvas.draw()

for k in range(3):
    # Get the 3 histogram axes for this column
    hist_axes = [fig.axes[k*3 + MM] for MM in range(3)]
    
    # Get the corresponding main plot axis (first row, same column)
    main_ax = axs[0][k]
    
    # Get main axis position in figure coordinates
    main_pos = main_ax.get_position()
    
    # Get current hist group left and right (from first and last hist in group)
    left_pos = hist_axes[0].get_position()
    right_pos = hist_axes[2].get_position()
    
    total_width = right_pos.x1 - left_pos.x0
    single_width = total_width / 3
    
    for MM in range(3):
        new_x0 = main_pos.x0 + MM * (main_pos.width / 3)
        new_x1 = new_x0 + main_pos.width / 3
        old_pos = hist_axes[MM].get_position()
        hist_axes[MM].set_position([new_x0, old_pos.y0, main_pos.width / 3, old_pos.height])
plt.show()

fig.savefig("figures/Fig_SI/structural_bias_comparison.pdf", dpi=300, bbox_inches='tight')